# Capability Lab — IP Exact-Head Gate v2
Launcher público. Cada gate roda em workspace isolado; nenhum código privado ou credencial é armazenado aqui.


In [ ]:
from google.colab import userdata
from getpass import getpass
import os, pathlib, subprocess

REPO = 'lucasmateus334-oss/lucas-capability-os'
BRANCH = 'governance/ip-independence-chain-of-title-v1'
WORK = pathlib.Path('/content/caplab-ip-gate')

def get_token():
    for key in ('GITHUB_TOKEN', 'CAPLAB_GITHUB_TOKEN', 'GH_TOKEN'):
        try:
            value = userdata.get(key)
            if value:
                return value
        except Exception:
            pass
    return getpass('GitHub token (não será exibido): ').strip()

token = get_token()
if not token:
    raise SystemExit('TOKEN_MISSING')

askpass = pathlib.Path('/content/caplab_askpass.sh')
askpass.write_text('#!/bin/sh\ncase \"$1\" in *Username*) echo \"x-access-token\";; *) printf \"%s\\n\" \"$CAPLAB_GH_TOKEN\";; esac\n', encoding='utf-8')
askpass.chmod(0o700)
env = os.environ.copy()
env['GIT_ASKPASS'] = str(askpass)
env['GIT_TERMINAL_PROMPT'] = '0'
env['CAPLAB_GH_TOKEN'] = token

try:
    if WORK.exists():
        subprocess.run(['rm', '-rf', str(WORK)], check=True)
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', f'https://github.com/{REPO}.git', str(WORK)], check=True, env=env)
    proc = subprocess.run(['python', 'scripts/run_ip_exact_head_colab_v2.py', '--push', '--branch', BRANCH], cwd=WORK, env=env)
    if proc.returncode != 0:
        raise SystemExit('CAPLAB_GATE_FAILED — procure a linha FAILED_STAGE acima')
    print('CAPLAB_COLAB_GATE=GREEN')
finally:
    env.pop('CAPLAB_GH_TOKEN', None)
    askpass.unlink(missing_ok=True)
    token = None

# O que isso faz: autentica apenas em runtime, clona o repo privado e executa cinco gates isolados; em falha mostra o estágio exato.
